# Data Cleaning & Processing Module

Reads raw CSVs from `data/raw/` and produces cleaned, feature-engineered files in `data/cleaned/`.

Pipeline steps:
1. **Load** raw price, financial statement, and macro data
2. **Deduplicate** — detect and remove duplicate rows with logging
3. **Missing values** — forward-fill trading gaps; drop only unfillable head rows
4. **Data-type normalisation** — dates parsed, numerics coerced
5. **Outlier detection** — flag single-day moves > ±50% (possible splits / data errors)
6. **Feature engineering** — daily returns, 7-day & 30-day MAs, 30-day volatility, Bollinger Bands
7. **Save** cleaned files

In [ ]:
import pandas as pd
import numpy as np
import warnings
import logging
import os

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(message)s")
log = logging.getLogger()

RAW_DIR     = "data/raw"
CLEANED_DIR = "data/cleaned"
os.makedirs(CLEANED_DIR, exist_ok=True)

print("Data Cleaning & Processing Pipeline")
print("=" * 50)

## 1. Load Raw Price Data

print("Loading raw price data...")
price_df = pd.read_csv(f"{RAW_DIR}/sp500_prices.csv")

# Guard: 'Ticker' missing means the CSV was saved with the wrong stack level (yfinance 1.x bug)
if "Ticker" not in price_df.columns:
    raise ValueError(
        f"'Ticker' column not found in sp500_prices.csv.\n"
        f"Actual columns: {price_df.columns.tolist()}\n"
        "The CSV was generated by run_collect.py with the wrong yfinance 1.x stack level.\n"
        "Re-run 'run_collect.py' (now fixed) and retry this notebook."
    )

# Normalise date — strip timezone if present
price_df["Date"] = pd.to_datetime(price_df["Date"], utc=True).dt.tz_localize(None)
price_df = price_df.sort_values(["Ticker", "Date"]).reset_index(drop=True)

print(f"Loaded : {price_df.shape[0]:,} rows | {price_df['Ticker'].nunique()} tickers")
print(f"Range  : {price_df['Date'].min().date()} → {price_df['Date'].max().date()}")
price_df.head()